# B cell FlowMap embedding

This notebook shows the exact embedding setup used to generate `flowmap_emb.pkl` from `bcell_velocity_standard.h5ad`. Hyperparameter exploration cells were removed so the reproduction path is visible.


In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import scanpy as sc
import scvelo as scv
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from scipy.sparse import issparse
import flowmap
from flowmap import *

# --------------------------
# 0. Load
# --------------------------

from pathlib import Path

cwd = Path.cwd()
ANALYSIS_DIR = (
    cwd
    if cwd.name == "06_b_cell_igvf"
    else Path("06_b_cell_igvf")
    if Path("06_b_cell_igvf").exists()
    else Path("..").resolve()
    if cwd.name == "notebooks"
    else Path("../..").resolve()
)
DATA_DIR = Path("data/flowmap_manuscript/b_cell")
if not DATA_DIR.exists():
    DATA_DIR = ANALYSIS_DIR.parent / "data" / "flowmap_manuscript" / "b_cell"
FIGURE_DIR = ANALYSIS_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

adata = sc.read_h5ad(DATA_DIR / "bcell_velocity_standard.h5ad")

# subset to HVGs
hvg_mask = adata.var["highly_variable"].values
X = adata.X[:, hvg_mask]
if issparse(X):
    X = X.toarray()

# V
V = adata.layers["velocity"][:, hvg_mask]
if issparse(V):
    V = V.toarray()

pseudotime = list(adata.obs["velocity_pseudotime"])

In [ ]:
import pickle

# This is the embedding configuration used for the saved manuscript B-cell FlowMap object.
emb = VectorFieldEmbedder(
    X,
    V,
    method="umap",
    dist_method="euclidean",
    dof=30,
    knn_k=30,
    embed_kwargs={
        "n_neighbors": 30,
        "min_dist": 0.6,
    },
)
emb.fit_embedding(seed=1)

fig = flowmap.plot.plot_velocity_grid(
    emb.X_emb,
    spline_vf=emb.spline_vf,
    grid_size=30,
    grid_density=1.0,
    smooth=0.5,
    n_neighbors=None,
    min_mass=0.01,
    scatter_color=pseudotime,
    scatter_size=10,
    scatter_alpha=0.1,
    arrow_color="black",
    arrow_alpha=0.9,
    arrow_scale=15.0,
    arrow_width=0.0025,
    figsize=(6, 6),
    cmap="viridis",
    vmin=None,
    vmax=None,
    show_axes=False,
    show_colorbar=False,
)

pickle.dump(emb, open(DATA_DIR / "flowmap_emb.pkl", "wb"))
